Aquí tienes el script definitivo que implementa la **Recomendación Metodológica Híbrida**. Este enfoque une lo mejor de los dos mundos para maximizar la precisión (buscar el MAE cercano a 5) bajo una arquitectura lineal robusta:

1. **La inercia epidemiológica se respeta de forma pura:** No sufre ninguna alteración ni reducción dimensional. El pasado directo de los casos de dengue se mantiene íntegro como regresor.
2. **El bloque climático se condensa mediante PCA:** Las 11 variables meteorológicas y sus 12 rezagos (143 columnas en total) se proyectan en unos pocos Componentes Principales ortogonales, eliminando el 100% de la multicolinealidad estructural y el ruido, evitando que el algoritmo colapse en mínimos locales como el $(0,1,0)$.
3. **Optimización por Grid Search sin mínimos locales:** Evaluamos el modelo híbrido resultante bajo los cuatro criterios tradicionales (`AICc`, `BIC`, `AIC`, `HQIC`) buscando la mejor combinación autorregresiva de la enfermedad potenciada por los componentes exógenos del clima.

```python


In [1]:
# =============================================================================
# PASO 1: IMPORTACIÓN DE LIBRERÍAS Y CONTROL DE ADVERTENCIAS
# =============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import pmdarima as pm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# =============================================================================
# PASO 2: CONFIGURACIÓN DE RUTAS Y CARGA DE DATOS
# =============================================================================
ruta_datos = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\8_recomendacion_estadistica_4\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx"
dir_resultados = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\8_recomendacion_estadistica_4\3_resultados"

os.makedirs(dir_resultados, exist_ok=True)

print(f"[INFO] Cargando dataset de alta dimensionalidad desde:\n{ruta_datos}")
df = pd.read_excel(ruta_datos)

# Configurar índice cronológico semanal ordinario
df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
df.set_index('fecha', inplace=True)
df = df.asfreq('W')
df = df.ffill().bfill()

# Identificar dinámicamente el rango temporal real
anio_min, anio_max = df.index.year.min(), df.index.year.max()
periodo_str = f"{anio_min}-{anio_max}"
print(f"[INFO] Periodo temporal unificado detectado: {periodo_str}")

# =============================================================================
# PASO 3: SEPARACIÓN METODOLÓGICA (INERCIA SAGRADA VS BLOQUE CLIMÁTICO)
# =============================================================================
y_target = df['casos_dengue']

# Separar las columnas que representan el pasado/inercia de la enfermedad
columnas_dengue_lags = [col for col in df.columns if 'casos_dengue' in col and col != 'casos_dengue']

# El bloque exógeno meteorológico puro (Variables climáticas + sus 12 rezagos)
columnas_excluir = ['casos_dengue', 'año', 'semana_epi', 'casos_ln'] + columnas_dengue_lags
columnas_meteorologicas = [col for col in df.columns if col not in columnas_excluir]

print(f"--> Inercia Directa del Dengue (Variables protegidas): {len(columnas_dengue_lags)}")
print(f"--> Bloque Climático total para colapsar vía PCA: {len(columnas_meteorologicas)}")

# =============================================================================
# PASO 4: REJILLA DE PARTICIONES CRONOLÓGICAS Y PIPELINE HÍBRIDO
# =============================================================================
particiones = {
    "95-5":  0.95,
    "96-4":  0.96,
    "97-3":  0.97
}

criterios = ['aicc', 'bic', 'aic', 'hqic']
resultados_globales = []

for criterio in criterios:
    print("\n" + "#"*85)
    print(f" PIPELINE HÍBRIDO: OPTIMIZACIÓN BAJO CRITERIO {criterio.upper()}")
    print("#"*85)
    
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 14), sharex=False, sharey=True)
    
    for idx, (nombre_split, tasa_train) in enumerate(particiones.items()):
        # 1. División temporal estricta de las series originales
        tamanio_train = int(len(df) * tasa_train)
        
        # Bloques de entrenamiento y prueba
        df_train = df.iloc[:tamanio_train]
        df_test = df.iloc[tamanio_train:]
        
        # 2. PROCESAMIENTO PCA EXCLUSIVO DEL BLOQUE CLIMÁTICO (Evitando fuga de información)
        scaler_clima = StandardScaler()
        X_train_clima_sc = scaler_clima.fit_transform(df_train[columnas_meteorologicas])
        X_test_clima_sc = scaler_clima.transform(df_test[columnas_meteorologicas])
        
        # Forzar un PCA que retenga de forma fija el 85% de la varianza del clima
        pca = PCA(n_components=0.85, random_state=42)
        X_train_pca = pca.fit_transform(X_train_clima_sc)
        X_test_pca = pca.transform(X_test_clima_sc)
        
        n_pcs = X_train_pca.shape[1]
        nombres_pcs = [f"PC_clima_{i+1}" for i in range(n_pcs)]
        
        df_train_pca = pd.DataFrame(X_train_pca, index=df_train.index, columns=nombres_pcs)
        df_test_pca = pd.DataFrame(X_test_pca, index=df_test.index, columns=nombres_pcs)
        
        # 3. RECONSTRUCCIÓN DE LA MATRIZ EXÓGENA FINAL (Inercia Dengue Lags + PCs del Clima)
        X_train_final = pd.concat([df_train[columnas_dengue_lags], df_train_pca], axis=1)
        X_test_final = pd.concat([df_test[columnas_dengue_lags], df_test_pca], axis=1)
        
        # Forzar frecuencias para statsmodels
        y_train, y_test = df_train['casos_dengue'], df_test['casos_dengue']
        y_train.index.freq = 'W'
        y_test.index.freq = 'W'
        X_train_final.index.freq = 'W'
        X_test_final.index.freq = 'W'
        
        # Escalado final de la matriz consolidada para estabilizar la máxima verosimilitud
        scaler_final = StandardScaler()
        X_train_final_sc = pd.DataFrame(scaler_final.fit_transform(X_train_final), index=X_train_final.index, columns=X_train_final.columns)
        X_test_final_sc = pd.DataFrame(scaler_final.transform(X_test_final), index=X_test_final.index, columns=X_test_final.columns)
        
        # 4. GRID SEARCH COMPLETO MEDIANTE AUTO_ARIMA (Búsqueda de la estructura real)
        modelo_auto = pm.auto_arima(
            y_train,
            X=X_train_final_sc,                
            start_p=1, max_p=4,       # Damos libertad a la inercia epidémica
            start_q=1, max_q=3,       
            d=1,                      
            seasonal=False,           
            stationary=False,
            information_criterion=criterio, 
            error_action='ignore',   
            suppress_warnings=True,  
            stepwise=False            # Grid Search Exhaustivo para romper el estancamiento (0,1,0)
        )
        
        p, d_ord, q = modelo_auto.order
        orden_opt = (p, d_ord, q)
        
        # 5. AJUSTE DE ESPACIO DE ESTADOS (SARIMAX)
        modelo_final = SARIMAX(
            y_train,
            exog=X_train_final_sc,
            order=orden_opt,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(method='lbfgs', maxiter=100, disp=False)
        
        # Extraer el valor del indicador del modelo ajustado
        if criterio == 'aicc': valor_metric = modelo_final.aicc
        elif criterio == 'bic': valor_metric = modelo_final.bic
        elif criterio == 'aic': valor_metric = modelo_final.aic
        else: valor_metric = modelo_final.hqic
        
        # 6. GENERACIÓN DE PRONÓSTICOS Y CÁLCULO DE ERRORES (MAE)
        y_train_pred = modelo_final.predict(start=y_train.index[0], end=y_train.index[-1], exog=X_train_final_sc)
        y_train_pred.iloc[:(d_ord + 1)] = np.nan
        
        y_test_pred = modelo_final.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test_final_sc)
        
        y_train_alined, y_train_pred_alined = y_train.dropna().align(y_train_pred.dropna(), join='inner')
        y_test_alined, y_test_pred_alined = y_test.dropna().align(y_test_pred.dropna(), join='inner')
        
        mae_train = mean_absolute_error(y_train_alined, y_train_pred_alined)
        mae_test = mean_absolute_error(y_test_alined, y_test_pred_alined)
        
        resultados_globales.append({
            "Criterio": criterio.upper(),
            "Partición": nombre_split,
            "Orden óptimo": f"({p},{d_ord},{q})",
            "Componentes PC": n_pcs,
            "Valor Criterio": valor_metric,
            "MAE Train": mae_train,
            "MAE Test": mae_test
        })
        
        # =========================================================================
        # PASO 5: DISEÑO DE GRÁFICOS INFORMATIVOS
        # =========================================================================
        ax_train = axes[idx, 0]
        ax_train.plot(y_train.index, y_train.values, label='Real Train', color='#1f77b4', alpha=0.8)
        ax_train.plot(y_train_pred.index, y_train_pred.values, label='Híbrido Pred', color='#ff7f0e', linestyle='--')
        ax_train.set_title(f"Ajuste Train ({nombre_split}) - Orden: {orden_opt} (MAE: {mae_train:.4f})", fontsize=10, fontweight='bold')
        ax_train.legend(loc='upper left', fontsize=9)
        
        ax_test = axes[idx, 1]
        ax_test.plot(y_test.index, y_test.values, label='Real Test', color='#2ca02c', alpha=0.8)
        ax_test.plot(y_test_pred.index, y_test_pred.values, label='Híbrido Forecast', color='#d62728', linestyle='--')
        ax_test.set_title(f"Pronóstico Test ({nombre_split}) (MAE: {mae_test:.4f})", fontsize=10, fontweight='bold')
        ax_test.legend(loc='upper left', fontsize=9)

    plt.suptitle(f'ARIMAX HÍBRIDO - Inercia Pura + PCA Climático (85% Var) | Criterio: {criterio.upper()}\nPeriodo Histórico: {periodo_str}', 
                 fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    ruta_grafico = os.path.join(dir_resultados, f"reporte_enfoque_hibrido_{criterio}_{periodo_str}.png")
    plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight')
    plt.close()

# =============================================================================
# PASO 6: CONSOLIDACIÓN TABULAR Y EXPORTACIÓN EXCEL FINAL
# =============================================================================
df_reporte_base = pd.DataFrame(resultados_globales)

lista_con_promedios = []
for crit, sub_df in df_reporte_base.groupby("Criterio"):
    lista_con_promedios.append(sub_df)
    
    fila_promedio = pd.DataFrame([{
        "Criterio": crit, "Partición": "PROMEDIO", "Orden óptimo": "-",
        "Componentes PC": int(round(sub_df["Componentes PC"].mean())),
        "Valor Criterio": sub_df["Valor Criterio"].mean(),
        "MAE Train": sub_df["MAE Train"].mean(), "MAE Test": sub_df["MAE Test"].mean()
    }])
    lista_con_promedios.append(fila_promedio)

df_reporte_completo = pd.concat(lista_con_promedios, ignore_index=True)

print("\n" + "="*95)
print(f"     REPORTE HISTÓRICO CONSOLIDADO: MODELACIÓN ENFOQUE HÍBRIDO ({periodo_str})     ")
print("="*95)
print(df_reporte_completo.to_string(index=False, formatters={
    "Valor Criterio": "{:.4f}".format, "MAE Train": "{:.4f}".format, "MAE Test": "{:.4f}".format
}))
print("="*95)

ruta_excel = os.path.join(dir_resultados, f"comparativa_enfoque_hibrido_pca_arima_{periodo_str}.xlsx")
df_reporte_completo.to_excel(ruta_excel, index=False)
print(f"\n[ÉXITO] Matriz de rendimientos del modelo híbrido grabada correctamente en:\n{ruta_excel}")


[INFO] Cargando dataset de alta dimensionalidad desde:
C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\1_reduccion_dimensional\8_recomendacion_estadistica_4\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx
[INFO] Periodo temporal unificado detectado: 2021-2025
--> Inercia Directa del Dengue (Variables protegidas): 12
--> Bloque Climático total para colapsar vía PCA: 143

#####################################################################################
 PIPELINE HÍBRIDO: OPTIMIZACIÓN BAJO CRITERIO AICC
#####################################################################################

#####################################################################################
 PIPELINE HÍBRIDO: OPTIMIZACIÓN BAJO CRITERIO BIC
#####################################################################################

#####################################################################################
 PIPELINE HÍBRIDO: OPTIMIZACIÓN BAJO CRITERIO AIC
##########################